## 1. Importações

In [11]:
# Manipulação de dados
import pandas as pd
import numpy as np
import time

# Visualização (para gráficos de performance)
import matplotlib.pyplot as plt
import seaborn as sns

# Algoritmos de Machine Learning
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Métricas de Avaliação
from sklearn.metrics import (
    classification_report, 
    roc_auc_score, 
    accuracy_score, 
    precision_score, 
    recall_score, 
    confusion_matrix
)

# Ferramentas extras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 2. Carregar Modelo

In [12]:

caminho_dados = '../data/processed/data_model/data_model.csv'
df_model = pd.read_csv(caminho_dados, sep=';')

print(f"Dataset carregado com sucesso! Formato atual: {df_model.shape}")

Dataset carregado com sucesso! Formato atual: (208919, 83)


### 2.1. Removendo vazamento de dados por variável `classificacao_acidente`

In [13]:
# Removendo variáveis de vazamento (Data Leakage) que entregam o gabarito
colunas_vazamento = [col for col in df_model.columns if 'classificacao_acidente' in col or 'tipo_envolvido' in col or 'estado_fisico' in col]

df_model = df_model.drop(columns=colunas_vazamento, errors='ignore')

# Agora sim, defina X e y
y = df_model['houve_obito']
X = df_model.drop(columns=['houve_obito'])

## 3. Variáveis Explicativas e Alvo

In [14]:
# Definindo o target e as features
y = df_model['houve_obito']
X = df_model.drop(columns=['houve_obito'])

print(f"Quantidade de variáveis preditoras (X): {X.shape[1]}")
print(f"Proporção do alvo:\n{y.value_counts(normalize=True) * 100}")

Quantidade de variáveis preditoras (X): 74
Proporção do alvo:
houve_obito
0    97.059626
1     2.940374
Name: proportion, dtype: float64


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Registros de Treino: {X_train.shape[0]}")
print(f"Registros de Teste:  {X_test.shape[0]}")

Registros de Treino: 167135
Registros de Teste:  41784


In [16]:
# Instanciando e aplicando o escalonador
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Padronização de recursos concluída.")

Padronização de recursos concluída.


In [17]:
# Configuração do modelo baseline com pesos balanceados
baseline_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

# Medindo o tempo de processamento do treinamento
inicio_treino = time.time()
baseline_model.fit(X_train_scaled, y_train)
tempo_processamento = time.time() - inicio_treino

print(f"Modelo baseline treinado em {tempo_processamento:.4f} segundos.")

Modelo baseline treinado em 0.8019 segundos.


In [18]:
# Gerando predições de classe e probabilidades
y_pred = baseline_model.predict(X_test_scaled)
y_prob = baseline_model.predict_proba(X_test_scaled)[:, 1]

# Computando as métricas de classificação
acuracia = accuracy_score(y_test, y_pred)
precisao = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
auc_roc = roc_auc_score(y_test, y_prob)

# Dicionário estruturado para armazenar os resultados e comparar posteriormente
historico_modelos = []
historico_modelos.append({
    "Modelo": "Regressão Logística (Baseline)",
    "Acurácia": acuracia,
    "Precisão": precisao,
    "Recall (Sensibilidade)": recall,
    "AUC-ROC": auc_roc,
    "Tempo (s)": round(tempo_processamento, 4)
})

# Exibindo o relatório detalhado do baseline
print("--- RELATÓRIO DE CLASSIFICAÇÃO (BASELINE) ---")
print(classification_report(y_test, y_pred))
print(f"Área sob a Curva ROC (AUC-ROC): {auc_roc:.4f}")

--- RELATÓRIO DE CLASSIFICAÇÃO (BASELINE) ---
              precision    recall  f1-score   support

           0       0.99      0.76      0.86     40555
           1       0.09      0.78      0.16      1229

    accuracy                           0.76     41784
   macro avg       0.54      0.77      0.51     41784
weighted avg       0.96      0.76      0.84     41784

Área sob a Curva ROC (AUC-ROC): 0.8552
